# 02 — EDA (eksploracyjna analiza danych)

**Cel notebooka:** zanim zbudujemy model, zrozumieć dane i sprawdzić, co naprawdę wpływa na wynik wyścigu. Wczytujemy wyłącznie pliki CSV z `01_dane` — bez API.

> ⚠️ **Nie uruchamiaj ponownie `01_dane`.** Wyścig w Austin już się odbył, więc API zwróciłoby jego wynik i trafiłby on do danych treningowych. Chcemy przewidywać Austin tak, jakby jeszcze się nie odbył (backtest), więc zostajemy przy zapisanych CSV.

## 2.1 Wczytanie danych

**Co robimy:** wczytujemy zapisane tabele i sprawdzamy ich rozmiary.

**Dlaczego:** każdy kolejny notebook zaczyna się od CSV, więc pierwsza kontrola to: czy wszystko się wczytało i czy rozmiary się zgadzają z tym, co widzieliśmy w `01_dane`.

In [1]:
import pandas as pd

races = pd.read_csv("data/races_2026.csv")
results = pd.read_csv("data/results_2026.csv")
quali = pd.read_csv("data/quali_2026.csv")
sprint = pd.read_csv("data/sprint_2026.csv")
fastest_laps = pd.read_csv("data/fastest_laps_2026.csv")
teammate_hist = pd.read_csv("data/teammate_gaps_hist.csv")
austin_hist = pd.read_csv("data/austin_hist.csv")

for name, df in [("races", races), ("results", results), ("quali", quali), ("sprint", sprint),
                 ("fastest_laps", fastest_laps), ("teammate_hist", teammate_hist), ("austin_hist", austin_hist)]:
    print(f"{name:15} {df.shape}")

races           (18, 17)
results         (374, 15)
quali           (391, 14)
sprint          (132, 15)
fastest_laps    (17, 7)
teammate_hist   (1370, 5)
austin_hist     (60, 8)


## 2.2 Czy pole position wygrywa?

**Co robimy:** dla każdego z 17 zakończonych wyścigów zestawiamy, kto stał na pole position i kto wygrał, i liczymy, jak często to ta sama osoba.

**Dlaczego:** to nasz punkt odniesienia (*baseline*). Jeśli „zwycięzcą będzie ten, kto ma pole" trafia w większości wyścigów, to model musi być lepszy od tej prostej reguły, żeby miał sens.

In [2]:
done_rounds = results["round"].unique()

pole = quali[(quali["round"].isin(done_rounds)) & (quali["position"] == 1)][["round", "name_acronym"]].rename(columns={"name_acronym": "pole"})
win = results[results["position"] == 1][["round", "name_acronym"]].rename(columns={"name_acronym": "winner"})

pole_vs_win = pole.merge(win, on="round")
pole_vs_win["pole_won"] = pole_vs_win["pole"] == pole_vs_win["winner"]

print(f"Pole position wygrało {pole_vs_win['pole_won'].sum()} z {len(pole_vs_win)} wyścigów ({pole_vs_win['pole_won'].mean():.0%})")
pole_vs_win

Pole position wygrało 12 z 17 wyścigów (71%)


,round,pole,winner,pole_won
0,1,RUS,RUS,True
1,2,ANT,ANT,True
2,3,ANT,ANT,True
3,4,ANT,ANT,True
4,5,RUS,ANT,False
5,6,ANT,ANT,True
6,7,RUS,HAM,False
7,8,RUS,RUS,True
8,9,ANT,LEC,False
9,10,ANT,ANT,True
